In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

!git clone https://github.com/manuelpalasanchez/tfm_deteccion.git
%cd tfm_deteccion


Mounted at /content/drive
Cloning into 'tfm_deteccion'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 94 (delta 37), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 29.77 KiB | 5.95 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/tfm_deteccion


In [4]:
!git pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 898 bytes | 449.00 KiB/s, done.
From https://github.com/manuelpalasanchez/tfm_deteccion
   1206e74..a1af953  main       -> origin/main
Updating 1206e74..a1af953
Fast-forward
 configs/base.yaml   |  7 +++----
 scripts/evaluate.py | 14 ++++++++------
 scripts/train.py    | 15 ++++++++-------
 3 files changed, 19 insertions(+), 17 deletions(-)


In [5]:
!pip install -r requirements.txt


In [6]:
import subprocess, os

drive_zips = "/content/drive/MyDrive/cnndetection-datasets"
cnn_root   = "/content/cnndetection"

# progan_train: 18 categorías (~63 GB, caben en los 69 GB libres)
destino_train = f"{cnn_root}/progan_train"
os.makedirs(destino_train, exist_ok=True)

cats_train = [
    "airplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa"
]

zip_train = f"{drive_zips}/progan_train.zip"
for cat in cats_train:
    print(f"[train] Extrayendo {cat}...")
    subprocess.run(["unzip", "-q", "-n", zip_train, f"{cat}/*",
                    "-d", destino_train], check=True)

print("✓ Train extraído —", end=" ")
!df -h /content | grep -v Filesystem

# progan_val: completo (pequeño)
destino_val = f"{cnn_root}/progan_val"
os.makedirs(destino_val, exist_ok=True)

print("Extrayendo val...")
subprocess.run(["unzip", "-q", "-n",
                f"{drive_zips}/progan_val.zip",
                "-d", destino_val], check=True)
print("✓ Val extraído")

print("\nEstructura:")
!ls /content/cnndetection/progan_train/ | head -5
!ls /content/cnndetection/progan_val/   | head -5


[train] Extrayendo airplane...
[train] Extrayendo bicycle...
[train] Extrayendo bird...
[train] Extrayendo boat...
[train] Extrayendo bottle...
[train] Extrayendo bus...
[train] Extrayendo car...
[train] Extrayendo cat...
[train] Extrayendo chair...
[train] Extrayendo cow...
[train] Extrayendo diningtable...
[train] Extrayendo dog...
[train] Extrayendo horse...
[train] Extrayendo motorbike...
[train] Extrayendo person...
[train] Extrayendo pottedplant...
[train] Extrayendo sheep...
[train] Extrayendo sofa...


CalledProcessError: Command '['unzip', '-q', '-n', '/content/drive/MyDrive/cnndetection-datasets/progan_train.zip', 'sofa/*', '-d', '/content/cnndetection/progan_train']' returned non-zero exit status 3.

In [7]:
import shutil
# Solo si motorbike quedó incompleta
shutil.rmtree("/content/cnndetection/progan_train/sofa", ignore_errors=True)

In [8]:
import subprocess, os

destino_val = "/content/cnndetection/progan_val"
os.makedirs(destino_val, exist_ok=True)

print("Extrayendo val...")
subprocess.run([
    "unzip", "-q", "-n",
    "/content/drive/MyDrive/cnndetection-datasets/progan_val.zip",
    "-d", destino_val
], check=True)

print("✓ Val listo")
!df -h /content | grep -v Filesystem
!ls /content/cnndetection/progan_val/ | head -5

Extrayendo val...
✓ Val listo
overlay         113G  107G  6.5G  95% /
airplane
bicycle
bird
boat
bottle


In [9]:
import yaml, pathlib

cfg_path = pathlib.Path("configs/base.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

cfg["data"]["train"]["root"]        = "/content/cnndetection"
cfg["data"]["val"]["root"]          = "/content/cnndetection"
cfg["data"]["eval_e1"]["root"]      = "/content/cnndetection"
cfg["data"]["eval_e1b"]["enabled"]  = False
cfg["data"]["eval_e2"]["enabled"]   = False
cfg["data"]["eval_e3"]["enabled"]   = False
cfg["data"]["num_workers"]          = 4
#cfg["training"]["batch_size"] = 128  # de 32 a 128, la GPU aguanta de sobra
#cfg["training"]["epochs"]     = 5    # de 10 a 5, suficiente para comparar arquitecturas


# Checkpoints a Drive (sobreviven si se cae la sesión)
cfg["output"]["base_dir"] = "/content/drive/MyDrive/tfm-checkpoints"

cfg_path.write_text(yaml.dump(cfg))
print("Config lista")


Config lista


In [11]:
modelo = "resnet50"  # cambiar a: freqnet | vit | universalfakedetect
!python scripts/train.py --config configs/{modelo}.yaml


09:11:23 INFO models.model_registry: Construyendo arquitectura 'resnet50' con kwargs={'pretrained': True}
09:11:38 INFO data.cnndetection_dataset: CNNDetectionDataset cargado - split='train' generadores=['progan'] muestras=612101
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
09:11:38 INFO data.cnndetection_dataset: CNNDetectionDataset cargado - split='val' generadores=['progan'] muestras=8000
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 3
wandb: You chose "Don't visualize 

In [ ]:
import glob

ckpts = glob.glob("/content/drive/MyDrive/tfm-checkpoints/*/checkpoint_best.pth")
print(f"Checkpoints en Drive: {len(ckpts)}")
for c in ckpts:
    print(" ", c)


EVALUACION

In [ ]:
import gdown, os, subprocess

cnn_root = "/content/cnndetection"
os.makedirs(cnn_root, exist_ok=True)

for zip_name in ["progan_testset.zip", "CNN_synth_testset.zip"]:
    fid = file_map[zip_name]
    zip_path = f"/content/{zip_name}"
    print(f"\nDescargando {zip_name}...")
    gdown.download(f"https://drive.google.com/uc?id={fid}", zip_path, quiet=False)
    print(f"Descomprimiendo {zip_name}...")
    subprocess.run(["unzip", "-q", "-n", zip_path, "-d", cnn_root], check=True)
    os.remove(zip_path)
    print(f"✓ {zip_name} listo")

print("\nCarpetas disponibles para evaluación:")
!ls /content/cnndetection/


In [ ]:
import os
carpetas = os.listdir("/content/cnndetection/")
print(carpetas)

# Si el testset se llamó "progan_testset" en vez de "progan_test", renombrar:
if "progan_testset" in carpetas and "progan_test" not in carpetas:
    os.rename("/content/cnndetection/progan_testset",
              "/content/cnndetection/progan_test")
    print("Renombrado progan_testset → progan_test")


In [ ]:
import yaml, pathlib

cfg_path = pathlib.Path("configs/base.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

cfg["data"]["eval_e1"]["root"]     = "/content/cnndetection"
cfg["data"]["eval_e1b"]["root"]    = "/content/cnndetection"
cfg["data"]["eval_e1b"]["enabled"] = True
cfg["data"]["eval_e2"]["enabled"]  = False
cfg["data"]["eval_e3"]["enabled"]  = False
cfg["data"]["num_workers"]         = 4
cfg["output"]["base_dir"]          = "/content/drive/MyDrive/tfm-checkpoints"

cfg_path.write_text(yaml.dump(cfg))
print("Config lista para evaluación")


In [ ]:
import glob

for modelo in ["resnet50", "freqnet", "vit", "universalfakedetect"]:
    ckpts = sorted(glob.glob(
        f"/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth"
    ))
    if not ckpts:
        print(f"⚠ Sin checkpoint para {modelo}, saltando")
        continue
    ckpt = ckpts[-1]
    print(f"\n{'='*50}\nEvaluando {modelo}\nCheckpoint: {ckpt}\n{'='*50}")
    !python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"
